In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("mecaniqa_dataset.csv", parse_dates=["Data"])
df = df.sort_values("Data").set_index("Data").asfreq("D")

# Tratamento básico para manter a série diária utilizável
df["Trocas_Oleo"] = df["Trocas_Oleo"].interpolate(method="time")
df["Manutencao_Motor"] = df["Manutencao_Motor"].interpolate(method="time")

print(df.head())
print("\nPeríodo:", df.index.min(), "até", df.index.max())

            Trocas_Oleo  Manutencao_Motor
Data                                     
2024-01-01         11.0               2.0
2024-01-02          9.0               4.0
2024-01-03         12.0               1.0
2024-01-04         15.0               8.0
2024-01-05         25.0              10.0

Período: 2024-01-01 00:00:00 até 2025-12-31 00:00:00


In [4]:
df_features = df.copy()

df_features["lag_1"] = df_features["Trocas_Oleo"].shift(1)
df_features["lag_7"] = df_features["Trocas_Oleo"].shift(7)
df_features["lag_30"] = df_features["Trocas_Oleo"].shift(30)

df_features["rolling_mean_7"] = (
    df_features["Trocas_Oleo"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

print("Antes da limpeza:")
print(df_features[["Trocas_Oleo", "lag_1", "lag_7", "lag_30", "rolling_mean_7"]].head(35))

Antes da limpeza:
            Trocas_Oleo  lag_1  lag_7  lag_30  rolling_mean_7
Data                                                         
2024-01-01         11.0    NaN    NaN     NaN             NaN
2024-01-02          9.0   11.0    NaN     NaN             NaN
2024-01-03         12.0    9.0    NaN     NaN             NaN
2024-01-04         15.0   12.0    NaN     NaN             NaN
2024-01-05         25.0   15.0    NaN     NaN             NaN
2024-01-06         25.0   25.0    NaN     NaN             NaN
2024-01-07         31.0   25.0    NaN     NaN             NaN
2024-01-08         13.0   31.0   11.0     NaN       18.285714
2024-01-09         10.0   13.0    9.0     NaN       18.571429
2024-01-10         13.0   10.0   12.0     NaN       18.714286
2024-01-11         10.0   13.0   15.0     NaN       18.857143
2024-01-12         25.0   10.0   25.0     NaN       18.142857
2024-01-13         28.0   25.0   25.0     NaN       18.142857
2024-01-14         22.0   28.0   31.0     NaN       

In [5]:
colunas_features = ["lag_1", "lag_7", "lag_30", "rolling_mean_7"]
df_final = df_features.dropna(subset=colunas_features).copy()

print("DataFrame final — validação QA com .head(15):")
print(df_final[["Trocas_Oleo"] + colunas_features].head(15))

print("\nValores nulos nas features:")
print(df_final[colunas_features].isna().sum())

DataFrame final — validação QA com .head(15):
            Trocas_Oleo  lag_1  lag_7  lag_30  rolling_mean_7
Data                                                         
2024-01-31         14.0   14.0   10.0    11.0       19.571429
2024-02-01         21.0   14.0   13.0     9.0       20.142857
2024-02-02         31.0   21.0   30.0    12.0       21.285714
2024-02-03         28.0   31.0   26.0    15.0       21.428571
2024-02-04         34.0   28.0   31.0    25.0       21.714286
2024-02-05         13.0   34.0   13.0    25.0       22.142857
2024-02-06         17.0   13.0   14.0    31.0       22.142857
2024-02-07         11.0   17.0   14.0    13.0       22.571429
2024-02-08         13.0   11.0   21.0    10.0       22.142857
2024-02-09         33.0   13.0   31.0    13.0       21.000000
2024-02-10         34.0   33.0   28.0    10.0       21.285714
2024-02-11         33.0   34.0   34.0    25.0       22.142857
2024-02-12         17.0   33.0   13.0    28.0       22.000000
2024-02-13         17.0 